## Workspace setup

In [2]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
from functools import partial
from termcolor import colored

import tensorflow as tf

## Data file loading

In [70]:
dataPath = '/home/akalinow/scratch/ELITPC/TPCReco/build/resources/'
files = [dataPath+'SimEvent_Track3D_TwoProng_gun_MC.root:TPCData']

stepSize = 3

fields = [
    "SimEvent/mySegments/mySegments.myStart",
    "SimEvent/mySegments/mySegments.myEnd",
    "Event/myChargeMap*",
    #"SimEvent.mySegments.myStart",
    #"mySegments.myEnd",
    #"Event/myChargeArray[3][3][256][512]",
    #"RecoEvent/mySegments/mySegments.myStart",
]

for array in uproot.iterate(files, step_size=stepSize, 
                                filter_name=fields, 
                                num_workers = 4, 
                                library="ak"):
        
        print("Fields in array: ", array.fields)
        branchName = "mySegments.myStart"
        # convert uproot awkward array to numpy array 
        fX = array[branchName]['fX'].to_numpy()
        print("fX shape: ", fX.shape)
        print("fX: ", fX[:10])  # Print first 10 elements for debugging
        break

TypeError: '<' not supported between instances of 'Model_tuple_3c_int_2c_int_2c_int_2c_int_3e__v1' and 'Model_tuple_3c_int_2c_int_2c_int_2c_int_3e__v1'

## Event loop

In [ ]:
%%time

batch_size = 32
event_in_batch = 1

for batch in chargeArray.iterate(step_size=batch_size, library="np"):
    
    eventData = batch["myChargeArray[3][3][256][512]"]
    eventData = np.sum(eventData, axis=2)
    eventData = eventData[event_in_batch]
    continue
    #Sum over sections - we do not use this information so far.
    #eventData = np.sum(eventData, axis=1)
      
    fig, ax = plt.subplots(1,3, figsize=(20,8))  
    im0 = ax[0].imshow(eventData[0,:,:], origin='lower')
    im1 = ax[1].imshow(eventData[1,:,:], origin='lower') 
    im1 = ax[2].imshow(eventData[2,:], origin='lower') 
    
    ax[0].set_title('U', fontsize = 15)
    ax[1].set_title('V', fontsize = 15) 
    ax[2].set_title('W', fontsize = 15) 
    break

In [ ]:
%%time
def UVW_projections_generator(imagesArray):
    for batch in chargeArray.iterate(step_size=1, library="np"):
        eventData = batch["myChargeArray[3][3][256][512]"]
        #Sum over sections - we do not use this information so far.
        eventData = np.sum(eventData, axis=2)
        #move projection dimension to the end
        eventData = np.moveaxis(eventData, 1, -1)
        yield eventData[0]
        
        
def labels_generator(labelsArray):
    for batch in labelsArray.iterate(step_size=1, library="np"):
        eventLabel = batch['track'][0][3]
        yield eventLabel/200      

batchSize = 64     
generator = partial(UVW_projections_generator, chargeArray)        
dataset_images = tf.data.Dataset.from_generator(
     generator,
     output_signature=(tf.TensorSpec(shape=(256, 512,3), dtype=tf.float32, name=None)))

generator = partial(labels_generator, labelsArray)        
dataset_labels = tf.data.Dataset.from_generator(
     generator,
     output_signature=(tf.TensorSpec(shape=(), dtype=tf.float32, name=None)))

dataset = tf.data.Dataset.zip((dataset_images, dataset_labels))
dataset = dataset.cache()
counter = 0
for item in dataset:
    counter +=1

print(counter)  

In [ ]:
from tensorflow.keras import datasets, layers, models

model = models.Sequential()
model.add(layers.Conv2D(32, (5, 5), activation='relu', input_shape=(256, 512,3)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (5, 5), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (5, 5), activation='relu'))
model.add(layers.Flatten())
model.add(layers.Dense(512, activation='relu'))
model.add(layers.Dense(1,activation='sigmoid'))


model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss="mape",
              metrics=['mse'])

model.fit(dataset.batch(16), epochs=1)
model.predict(dataset.batch(1).take(2))
for item in dataset.batch(1).take(2):
    print(item[1])